[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sk-2103/AI4Sustainability-Labs/blob/main/labs/lab04_cnn_surface_water/lab04_evaluation.ipynb)  
**AI for Sustainability** · University of Wisconsin–Madison · Labs by Saurabh Kaushik · Course instructor: Dr. Beth Tellman  
*Make a copy (File → Save a copy in Drive) before you start. Part of the [AI for Sustainability lab series](https://github.com/Sk-2103/AI4Sustainability-Labs).*

# 🌊 Evaluation: CNNs for Surface Water Mapping
## AI for Sustainability — Lab 4 Assessment

---

**Student Name:** ___________________________  &nbsp;&nbsp;&nbsp; **Student ID:** _______________  &nbsp;&nbsp;&nbsp; **Date:** _______________

---

### 📋 Instructions
- This evaluation is based on **Lab 4: CNN Water Detection** that you completed.
- Total marks: **25** (Theory: 13 | Practical: 12)
- For **fill-in-the-blank** code cells, only edit the sections marked `# YOUR CODE HERE`.
- **Do not modify** any other code unless instructed.
- Run all cells **in order** from top to bottom.
- Refer back to your lab notebook for hints — understanding, not memorization, is the goal.

> 📗 **Reference:** Your lab notebook covers all concepts tested here.  
> Key sections: *Part 1 (How Satellites See Water)*, *Part 2 (CNN concepts)*, *Sections 4–7 (NDWI, TinySegCNN, U-Net)*.

---




---
## ⚙️ Setup — Run This First
> Run the cells below to install packages and load the dataset. You do not need to modify anything here.


In [ ]:
# Install dependencies (same as your lab notebook — safe to skip if already installed)
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torchgeo', 'rasterio', 'scikit-learn', 'scipy']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)
print('✅ All packages ready')

In [ ]:
import os, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchgeo.datasets import RasterDataset, stack_samples, unbind_samples, utils
from torchgeo.samplers import RandomGeoSampler, Units

print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Download and load dataset (same as lab notebook)
DATA_ROOT = Path("/content/surface_water")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_url = "https://hf.co/datasets/cordmaur/earth_surface_water/resolve/main/earth_surface_water.zip"
utils.download_and_extract_archive(zip_url, DATA_ROOT)
root = DATA_ROOT / "dset-s2"

def scale(item):
    item["image"] = item["image"] / 10000.0
    return item

train_imgs = RasterDataset(paths=(root / "tra_scene").as_posix(), crs="epsg:3395", res=10, transforms=scale)
train_msks = RasterDataset(paths=(root / "tra_truth").as_posix(), crs="epsg:3395", res=10)
valid_imgs = RasterDataset(paths=(root / "val_scene").as_posix(), crs="epsg:3395", res=10, transforms=scale)
valid_msks = RasterDataset(paths=(root / "val_truth").as_posix(), crs="epsg:3395", res=10)
train_msks.is_image = False
valid_msks.is_image = False

train_dset = train_imgs & train_msks
valid_dset = valid_imgs & valid_msks

PATCH = 256
train_sampler = RandomGeoSampler(train_imgs, size=PATCH, length=96, units=Units.PIXELS)
valid_sampler = RandomGeoSampler(valid_imgs, size=PATCH, length=32, units=Units.PIXELS)

train_loader = DataLoader(train_dset, sampler=train_sampler, batch_size=4, collate_fn=stack_samples, num_workers=0)
valid_loader = DataLoader(valid_dset, sampler=valid_sampler, batch_size=4, collate_fn=stack_samples, num_workers=0)

# Helper: fix mask shape and normalise to 0/1
def batch_xy(b):
    x = b["image"].float()
    y = b["mask"]
    if y.ndim == 3:
        y = y.unsqueeze(1)
    y = y.float()
    if y.max() > 1:
        y = (y > 0).float()
    return x, y

batch = next(iter(train_loader))
C = batch["image"].shape[1]
print(f"✅ Dataset loaded | image shape: {batch['image'].shape} | {C} spectral bands")

---
# 📝 Part A — Theory Questions  *(13 marks)*

Answer the questions in the Markdown cells provided below each question.  
Double-click a Markdown cell to edit it, then press **Shift+Enter** to render.


---
### Q1 — Satellite Bands & CNN *(8 marks)*

> 💡 Refer to **Part 1 of your lab notebook** (sections 1.1 and 1.2).



**(a)** (2 mark) what is the difference between validation and test score?

 *Your answer:* ___________


**(b)** (2 mark) Why does **open water appear very dark** in the Near-Infrared (NIR) band?  
Write 1–2 sentences:

> *Your answer:* ___________

**(b)** (2 mark) What is maxpooling?
Write 1–2 sentences:

> *Your answer:* ___________

**(b)** (2 mark) What is the difference between apoch and iteration?
Write 1–2 sentences:

> *Your answer:* ___________


---
### Q2 — CNN Core Operations *(5 marks)*

> 💡 Refer to **Part 2 (sections 2.1 and 2.2) of your lab notebook**.

Match each CNN operation to its correct description. Write the letter next to the number.

**Operations:**  
1. Convolution  
2. ReLU  
3. Batch Normalization  
4. Max Pooling  
5. Skip Connections (U-Net)  

**Descriptions:**  
(A) Slides a small grid of weights over the image to detect local patterns  
(B) Sets all negative values to zero, introducing non-linearity  
(C) Takes the maximum value in each small window, reducing spatial size  
(D) Rescales activations to have mean ≈ 0 and std ≈ 1 for training stability  
(E) Passes high-resolution detail from encoder directly to decoder  

**Your answers:**

| # | Letter |
|---|--------|
| 1 | ___ |
| 2 | ___ |
| 3 | ___ |
| 4 | ___ |
| 5 | ___ |


---
# 💻 Part B — Practical (Coding) Questions  *(12 marks)*

Fill in the blanks marked `# YOUR CODE HERE`.  
Each blank is a **single line or expression** unless told otherwise.  
Run each cell after completing it to check your output.


---
### P1 — Compute NDWI *(2.5 marks)*

> 💡 Refer to `compute_ndwi()` in **Section 5 of your lab notebook**.

Complete the `compute_ndwi` function below.  
Band indices for this dataset: **Green = index 1, NIR = index 3**.


In [ ]:
# ── P1: Compute NDWI ─────────────────────────────────────────────── [2 marks]

def compute_ndwi(x_chw, green_idx=1, nir_idx=3, eps=1e-6):
    """
    Compute NDWI = (Green - NIR) / (Green + NIR + eps)

    Args:
        x_chw : Tensor of shape (C, H, W) — one image patch
        green_idx : channel index for Green band
        nir_idx   : channel index for NIR band
        eps       : small constant to avoid division by zero

    Returns:
        ndwi : Tensor of shape (H, W)
    """
    # Step 1: extract the green and NIR bands as float tensors
    green = x_chw[green_idx].float()    # ← already done for you
    nir   = # YOUR CODE HERE            # (0.5 mark) extract NIR band

    # Step 2: apply the NDWI formula
    ndwi  = # YOUR CODE HERE            # (1 mark)   (green - nir) / (green + nir + eps)

    return ndwi


# ── Test your function ──────────────────────────────────────────────────────
# Grab one sample from the training loader
sample_batch = next(iter(train_loader))
x_test = sample_batch["image"][0].float()   # shape: (C, H, W)

ndwi_result = compute_ndwi(x_test)

# (1 mark) Apply a threshold of 0.0 to create a binary water mask (True = water)
water_mask = # YOUR CODE HERE

print("NDWI shape:", ndwi_result.shape)
print("NDWI min:", round(float(ndwi_result.min()), 3),
      "| max:", round(float(ndwi_result.max()), 3))
print("Water pixels (%):", round(float(water_mask.float().mean()) * 100, 2))

# Visualise
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].imshow(ndwi_result.numpy(), cmap="RdYlGn")
axs[0].set_title("NDWI map"); axs[0].axis("off")
axs[1].imshow(water_mask.numpy(), cmap="Blues")
axs[1].set_title("Water mask (thr=0.0)"); axs[1].axis("off")
plt.tight_layout(); plt.show()

---
### P2 — Complete the TinySegCNN *(3 marks)*

> 💡 Refer to `TinySegCNN` in **Section 6 of your lab notebook**.

The model architecture is shown in the diagram below.  
Fill in the missing layers to complete the `forward` method and `__init__`.

```
Input (C bands, H, W)
        │
   Conv2d(C → 32, 3×3, pad=1) + ReLU
        │
   Conv2d(32 → 32, 3×3, pad=1) + ReLU     ← you complete this
        │
   Conv2d(32 → 32, 3×3, pad=1) + ReLU
        │
   Conv2d(32 → 1, 1×1)                     ← output: 1 logit per pixel
        │
  Output (1, H, W)
```


In [ ]:
# ── P2: Complete TinySegCNN ──────────────────────────────────────── [3 marks]

class TinySegCNN(nn.Module):
    def __init__(self, in_ch: int, base: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            # Layer 1: conv in_ch → base channels, 3×3 kernel, padding=1
            nn.Conv2d(in_ch, base, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Layer 2 (1 mark): conv base → base channels, 3×3 kernel, padding=1
            # YOUR CODE HERE (nn.Conv2d ...),
            # YOUR CODE HERE (nn.ReLU  ...),

            # Layer 3: conv base → base channels, 3×3 kernel, padding=1
            nn.Conv2d(base, base, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Layer 4 (1 mark): 1×1 conv to reduce to 1 output channel (the water logit)
            # YOUR CODE HERE
        )

    def forward(self, x):
        # (1 mark): pass x through self.net and return the result
        return # YOUR CODE HERE


# ── Sanity check — run this to verify shapes ────────────────────────────────
tiny = TinySegCNN(in_ch=C).to(device)
x_dummy = torch.zeros(2, C, 64, 64).to(device)
out = tiny(x_dummy)
print("Input shape :", x_dummy.shape)
print("Output shape:", out.shape)   # expected: torch.Size([2, 1, 64, 64])
assert out.shape == (2, 1, 64, 64), "❌ Output shape incorrect — check your layers!"
print("✅ Shape check passed!")

---
### P3 — Training Loop *(3 marks)*

> 💡 Refer to the training loop in **Section 6 of your lab notebook**.

Complete the training loop below. The optimizer and loss function are already defined for you.


In [ ]:
# ── P3: Training loop ────────────────────────────────────────────── [3 marks]

# Loss function and optimiser — already defined, do not change
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(tiny.parameters(), lr=1e-3)

# IoU evaluation helper — already defined, do not change
def iou_score(pred, target, eps=1e-6):
    pred   = pred.bool()
    target = target.bool()
    inter  = (pred & target).sum().item()
    union  = (pred | target).sum().item()
    return inter / (union + eps)

@torch.no_grad()
def eval_iou(model, loader, n_batches=5):
    model.eval()
    scores = []
    for i, b in enumerate(loader):
        if i >= n_batches: break
        x, y = batch_xy(b)
        x, y = x.to(device), y.to(device)
        pred = (torch.sigmoid(model(x)) > 0.5)
        scores.append(iou_score(pred.squeeze(1), y.squeeze(1)))
    return float(np.mean(scores)) if scores else float("nan")


EPOCHS = 10

for ep in range(1, EPOCHS + 1):
    tiny.train()
    epoch_losses = []

    for b in train_loader:
        # (0.5 mark) Unpack the batch into x (images) and y (masks)
        x, y = # YOUR CODE HERE          # hint: use batch_xy(b)

        # Move tensors to the correct device
        x, y = x.to(device), y.to(device)

        # (0.5 mark) Zero out the gradients from the previous step
        # YOUR CODE HERE

        # (0.5 mark) Forward pass: compute model predictions (logits)
        logits = # YOUR CODE HERE

        # (0.5 mark) Compute the loss between logits and ground-truth mask y
        loss = # YOUR CODE HERE

        # (0.5 mark) Backward pass: compute gradients
        # YOUR CODE HERE

        # (0.5 mark) Update model weights
        # YOUR CODE HERE

        epoch_losses.append(loss.item())

    val_iou = eval_iou(tiny, valid_loader)
    print(f"Epoch {ep:02d} | loss = {np.mean(epoch_losses):.4f} | val IoU = {val_iou:.3f}")

print("\n✅ Training complete!")

---
### P4 — Evaluate & Compare *(4 marks)*

> 💡 Refer to **Section 5 (NDWI IoU)** and **Section 6/7 (model visualisation)** in your lab notebook.

Run the comparison cell below and then answer the written questions underneath.


In [ ]:
# ── P4a: Compute IoU for NDWI baseline ──────────────────────────── [1 mark]

# Collect a few validation samples for comparison
def collect_samples(loader, n=6, min_frac=0.05, max_b=300):
    xs, ys = [], []
    for b in loader:
        x, y = batch_xy(b)
        y_hw = y[:, 0]
        wf = (y_hw > 0).float().mean(dim=(1, 2))
        for i in range(x.shape[0]):
            if wf[i] >= min_frac:
                xs.append(x[i].cpu()); ys.append(y_hw[i].cpu())
            if len(xs) >= n: return xs, ys
    return xs, ys

xs, ys = collect_samples(valid_loader, n=6)

# Compute NDWI IoU across the collected samples
ndwi_ious = []
for x0, y0 in zip(xs, ys):
    # (0.5 mark): compute ndwi using your function from P1
    ndwi = # YOUR CODE HERE
    # (0.5 mark): apply threshold 0.0 to get binary prediction
    pred_ndwi = # YOUR CODE HERE
    ndwi_ious.append(iou_score(pred_ndwi, y0 > 0))

print(f"NDWI mean IoU : {np.mean(ndwi_ious):.3f}")

In [ ]:
# ── P4b: Compute IoU for TinySegCNN and visualise ────────────────── [2 marks]

tiny.eval()
tiny_ious = []

fig, axs = plt.subplots(len(xs), 4, figsize=(16, 4 * len(xs)))

for i, (x0, y0) in enumerate(zip(xs, ys)):

    # TinySegCNN prediction
    with torch.no_grad():
        prob = torch.sigmoid(tiny(x0.unsqueeze(0).to(device)))[0, 0].cpu()
    pred_tiny = (prob > 0.5)
    tiny_ious.append(iou_score(pred_tiny, y0 > 0))

    # NDWI prediction
    ndwi     = compute_ndwi(x0)
    pred_ndw = (ndwi > 0.0)

    # RGB for display
    rgb = x0[[3, 2, 1]].float()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    rgb = rgb.permute(1, 2, 0).numpy()

    axs[i, 0].imshow(rgb);                  axs[i, 0].set_title("RGB")
    axs[i, 1].imshow(pred_ndw.numpy());     axs[i, 1].set_title(f"NDWI  IoU={ndwi_ious[i]:.2f}")
    axs[i, 2].imshow(pred_tiny.numpy());    axs[i, 2].set_title(f"TinyCNN IoU={tiny_ious[i]:.2f}")
    axs[i, 3].imshow(y0.numpy(), cmap="Blues"); axs[i, 3].set_title("Ground Truth")
    for j in range(4): axs[i, j].axis("off")

plt.tight_layout()
plt.show()

print(f"TinySegCNN mean IoU : {np.mean(tiny_ious):.3f}")
print(f"NDWI       mean IoU : {np.mean(ndwi_ious):.3f}")

#### P4c — Written Reflection *(1 mark)*

Answer both questions below (1–2 sentences each):

**(i)** (0.5 mark) Based on your IoU results, which method performed better — NDWI or TinySegCNN? Give one reason *why* the better method might outperform the other.

> *Your answer:* ___________

**(ii)** (0.5 mark) Look at the visualisations. Describe **one specific type of error** you can see in either the NDWI or TinySegCNN predictions (e.g., a region that was wrongly detected, or a water body that was missed). Why might this error occur?

> *Your answer:* ___________


---
## ✅ Submission Checklist

Before submitting, confirm all of the following:

- [ ] All theory questions (Q1–Q5) are answered in the Markdown cells
- [ ] `compute_ndwi()` runs without errors and the plot looks correct (P1)
- [ ] `TinySegCNN` shape check prints ✅ (P2)
- [ ] Training loop prints loss and IoU for each epoch (P3)
- [ ] Comparison visualisation shows 6 rows with RGB / NDWI / TinyCNN / GT (P4)
- [ ] Written reflection (P4c) is filled in
- [ ] The notebook runs **top-to-bottom** without errors

**Save your notebook:** File → Download → Download .ipynb  
Then upload to the course portal.

---
*Good luck! Remember — understanding the concepts matters more than getting perfect numbers.* 🌊
